In [1]:
 def get_pivot(row):
    for idx, val in enumerate(row):
        if val != 0:
            return idx
    return None

def copy(matrix):
    return [row[:] for row in matrix]
    
def generalized_echelon_form_GF2(matrix):
    m_copy = copy(matrix)
    rows = len(m_copy)
    cols = len(m_copy[0])
    operations = []
    lead = 0

    for r in range(rows):
        if lead >= cols:
            break
        i = r
        while m_copy[i][lead] == 0:
            i += 1
            if i == rows:
                i = r
                lead += 1
                if lead == cols:
                    return m_copy, operations
        m_copy[r], m_copy[i] = m_copy[i], m_copy[r]
        if r != i:
            operations.append((r, i))
            operations.append((i, r))
            operations.append((r, i))
        for i in range(rows):
            if i != r and m_copy[i][lead] == 1:
                m_copy[i] = [(x + y) % 2 for x, y in zip(m_copy[i], m_copy[r])]
                operations.append((i, r))
        lead += 1

    return m_copy, operations

def print_matrix(matrix):
    for row in matrix:
        print(" ".join(map(str, row)))

In [2]:
from algebraic_immunity_utils import Matrix as GF2Matrix

In [3]:
import random

n = 4
M = [[random.randint(0,1) for _ in range(n)] for _ in range(n)]
print_matrix(M)

1 0 0 1
0 0 1 0
0 1 0 1
0 1 1 1


In [4]:
m , ops = generalized_echelon_form_GF2(M)
print_matrix(m)
print(ops)
ms = GF2Matrix(M)
mes, opsr = ms.row_echelon_full_matrix()
print_matrix(mes.to_list())
print(opsr)

1 0 0 1
0 1 0 1
0 0 1 0
0 0 0 0
[(1, 2), (2, 1), (1, 2), (3, 1), (3, 2)]
1 0 0 1
0 1 0 1
0 0 1 0
0 0 0 0
[(1, 2), (2, 1), (1, 2), (3, 1), (3, 2)]


In [5]:
n = 1000
for _ in range(1):
    M = [[random.randint(0,1) for _ in range(n)] for _ in range(n)]
    ms = GF2Matrix(M)
    mes, opsr = ms.row_echelon_full_matrix()
    # m, ops = generalized_echelon_form_GF2(M)
    G = Matrix(GF(2), M)
    assert G.rref() == Matrix(GF(2), mes.to_list())
    

In [10]:
def echf_3(m):
    m_copy = copy(m)  # Assuming m is an object that has a `copy()` method
    last_row_index = m.nrows() - 1
    last_row = m_copy[-1]
    operations = []

    # Check for pivots and perform operations on the last row
    for col_index in range(m_copy.ncols()):
        p_index = get_pivot(last_row)  # This should find the pivot in the last row

        if p_index is None:
            print(m_copy)
            for row_index in range(1, m_copy.nrows() - 1)[::-1]:
                if m_copy[row_index].is_zero():
                    continue
                curr_pivot = get_pivot(m_copy[row_index])
                if curr_pivot == row_index:
                    break
                prev_pivot = get_pivot(m_copy[row_index - 1])
                # print(curr_pivot, prev_pivot)
                print()
                if prev_pivot is None or (prev_pivot is not None and curr_pivot < prev_pivot):
                    m_copy[row_index], m_copy[row_index - 1] = m_copy[row_index - 1], m_copy[row_index]
                    operations.append((row_index, row_index + 1))
                    operations.append((row_index + 1, row_index))
                    operations.append((row_index, row_index + 1))
                elif prev_pivot is not None and prev_pivot == curr_pivot == m_copy.ncols() - 1:
                    print(curr_pivot, prev_pivot)
                    m_copy[row_index] = m_copy[row_index] + m_copy[row_index - 1]
                    operations.append((row_index, row_index - 1))
            print("HERE Just")
            break
        else:
            p_row = None
            j_index = None

            # Look for a row above that has the pivot at p_index
            d_base = m_copy.nrows() - 1
            closest = None
            for j in range(m_copy.nrows() - 1):  # Go up to the second-to-last row
                piv = get_pivot(m_copy[j])
                if piv is not None:
                    if piv == p_index:
                        p_row = m_copy[j]
                        j_index = j
                        break
                    else:
                        d = piv - p_index
                        if 0 < d < d_base:
                            closest = j
                            d_base = d
                else:
                    if closest is None:
                        closest = j
                    print("HERE")
                    break

            if p_row is None:  # No row found with a pivot at p_index
                m_copy[-1], m_copy[closest] = m_copy[closest], m_copy[-1]
                last_row = m_copy[-1]  # Update the last row after the swap
                operations.append((closest, last_row_index))
                operations.append((last_row_index, closest))
                operations.append((closest, last_row_index))

            elif p_row[p_index] == 1:
                m_copy[-1] = m_copy[-1] + p_row
                last_row = m_copy[-1]
                operations.append((last_row_index, j_index))
                new_pivot = get_pivot(last_row)
                if new_pivot is not None:
                    for r in range(m_copy.nrows()):
                        if m_copy[r][new_pivot] == 1:
                            m_copy[r] = m_copy[r] + m_copy[last_row]

    return m_copy, operations


def echf_1(m):
    m_copy = copy(m)  # Assuming m is an object that has a `copy()` method
    last_row_index = m.nrows() - 1
    last_row = m_copy[-1]
    operations = []

    # Check for pivots and perform operations on the last row
    for col_index in range(m_copy.ncols()):
        p_index = get_pivot(last_row)  # This should find the pivot in the last row

        if p_index is None:
            print(m_copy)
            for row_index in range(1, m_copy.nrows() - 1)[::-1]:
                if m_copy[row_index].is_zero():
                    continue
                curr_pivot = get_pivot(m_copy[row_index])
                if curr_pivot == row_index:
                    break
                prev_pivot = get_pivot(m_copy[row_index - 1])
                # print(curr_pivot, prev_pivot)
                print()
                if prev_pivot is None or (prev_pivot is not None and curr_pivot < prev_pivot):
                    m_copy[row_index], m_copy[row_index - 1] = m_copy[row_index - 1], m_copy[row_index]
                    operations.append((row_index, row_index + 1))
                    operations.append((row_index + 1, row_index))
                    operations.append((row_index, row_index + 1))
                elif prev_pivot is not None and prev_pivot == curr_pivot == m_copy.ncols() - 1:
                    print(curr_pivot, prev_pivot)
                    m_copy[row_index] = m_copy[row_index] + m_copy[row_index - 1]
                    operations.append((row_index, row_index - 1))

            print("HERE Just")
            break
            # swap_row_index = None
            # for row_index in range(m_copy.nrows() - 1)[::-1]:
            #     if m_copy[row_index][row_index] == 0 and any(
            #             m_copy[row_index][k] == 1 for k in range(row_index + 1, m_copy.nrows())):
            #         swap_row_index = row_index
            # if swap_row_index is not None and not m_copy[swap_row_index+1].is_zero() :
            #     m_copy[swap_row_index], m_copy[swap_row_index + 1] = m_copy[swap_row_index + 1], m_copy[swap_row_index]
            #     operations.append((swap_row_index, swap_row_index + 1))
            #     operations.append((swap_row_index + 1, swap_row_index))
            #     operations.append((swap_row_index, swap_row_index + 1))
            # else:
            #     break

        else:
            p_row = None
            j_index = None

            # Look for a row above that has the pivot at p_index
            d_base = m_copy.nrows() - 1
            closest = None
            for j in range(m_copy.nrows() - 1):  # Go up to the second-to-last row
                piv = get_pivot(m_copy[j])
                if piv is not None:
                    if piv == p_index:
                        p_row = m_copy[j]
                        j_index = j
                        break
                    else:
                        d = piv - p_index
                        if 0 < d < d_base:
                            closest = j
                            d_base = d
                else:
                    if closest is None:
                        closest = j
                    print("HERE")
                    break
                # if m_copy[j][p_index] == 1 and not any(m_copy[j][k] == 1 for k in range(p_index)):
                #    p_row = m_copy[j]
                #    j_index = j
                #    break

            if p_row is None:  # No row found with a pivot at p_index
                # if p_index == last_row_index:
                #     swap_index_row = None
                #     for r in range(m_copy.nrows() - 1):
                #         if m_copy[r].is_zero():  # Check if row is zero
                #             swap_index_row = r
                #             break
                #     if swap_index_row is not None:
                #         m_copy[-1], m_copy[swap_index_row] = m_copy[swap_index_row], m_copy[-1]
                #         operations.append((swap_index_row, last_row_index))
                #         operations.append((last_row_index, swap_index_row))
                #         operations.append((swap_index_row, last_row_index))
                #     break  # Exit if we can't find a valid pivot
                # Swap rows to bring the pivot to the last row
                m_copy[-1], m_copy[closest] = m_copy[closest], m_copy[-1]
                last_row = m_copy[-1]  # Update the last row after the swap
                operations.append((closest, last_row_index))
                operations.append((last_row_index, closest))
                operations.append((closest, last_row_index))

            elif p_row[p_index] == 1:
                m_copy[-1] = m_copy[-1] + p_row
                last_row = m_copy[-1]
                operations.append((last_row_index, j_index))

    return m_copy, operations

In [16]:
M = Matrix(GF(2), 
          [
              [1,0, 1],
              [1,1,0]
          ]
          )

echf_1(M)

TypeError: index must be an integer